# Video Classification with InternVideo2 and OpenVINO

InternVideo2 is family of video foundation models (ViFM) that achieve the state-of-the-art results in video recognition, video-text tasks, and video-centric dialogue.
You can find more information about model in [model card](https://huggingface.co/OpenGVLab/InternVideo2-Stage2_6B), [paper](https://arxiv.org/pdf/2403.15377) and original [repository](https://github.com/OpenGVLab/InternVideo/tree/main/InternVideo2/multi_modality).

In this tutorial we consider how to convert, optimize and run InternVideo2 Stage2 model for video classification using OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert model to OpenVINO Intermediate Representation](#Convert-model-to-OpenVINO-Intermediate-Representation)
    - [Compress model weights](#Compress-model-weights)
- [Prepare model inference pipeline](#Prepare-model-inference-pipeline)
    - [Select inference device](#Select-inference-device)
- [Run model inference](#Run-model-inference)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/intern-video2-classiciation/intern-video2-classification.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import platform

%pip install -q "torch>=2.1" "torchvision" "opencv-python" "transformers>=4.45" "einops>=0.7.0" "timm>=0.5.4" "gradio>=4.19" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "openvino>=2025.1.0" "nncf>=2.16.0"

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0"

In [2]:
import requests
from pathlib import Path

if not Path("ov_internvideo_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/intern-video2-classiciation/ov_internvideo_helper.py"
    )
    open("ov_internvideo_helper.py", "w").write(r.text)

if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/intern-video2-classiciation/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

## Convert model to OpenVINO Intermediate Representation
[back to top ⬆️](#Table-of-contents:)


InternVideo2 is PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). [OpenVINO model conversion API](https://docs.openvino.ai/2024/openvino-workflow/model-preparation.html#convert-a-model-with-python-convert-model) should be used for these purposes. `ov.convert_model` function accepts original PyTorch model instance and example input for tracing and returns `ov.Model` representing this model in OpenVINO framework. Converted model can be used for saving on disk using `ov.save_model` function or directly loading on device using `core.complie_model`.

Model consist of 2 parts:
* **Vision Encoder** for conversion video frames to embeddings space
* **Text Encoder** for conversion text labels to embeddings space

Model performs text-to-video retrieval task comparing similarity between text and vision features. For preserving original model flexibility, we will convert each part separately.
The script `ov_internvideo_helper.py` contains helper function for model conversion, please check its content if you interested in conversion details.

In [3]:
from ov_internvideo_helper import convert_internvideo

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("intern-video2-classification.ipynb")

# Uncomment the line to see model conversion code
# ??convert_internvideo

### Compress model weights
[back to top ⬆️](#Table-of-contents:)

For reducing memory consumption, weights compression optimization can be applied using [NNCF](https://github.com/openvinotoolkit/nncf). 

<details>
    <summary><b>Click here for more details about weight compression</b></summary>
Weight compression aims to reduce the memory footprint of a model. It can also lead to significant performance improvement for large memory-bound models, such as Large Language Models (LLMs). LLMs and other models, which require extensive memory to store the weights during inference, can benefit from weight compression in the following ways:

* enabling the inference of exceptionally large models that cannot be accommodated in the memory of the device;

* improving the inference performance of the models by reducing the latency of the memory access when computing the operations with weights, for example, Linear layers.

[Neural Network Compression Framework (NNCF)](https://github.com/openvinotoolkit/nncf) provides 4-bit / 8-bit mixed weight quantization as a compression method primarily designed to optimize LLMs. The main difference between weights compression and full model quantization (post-training quantization) is that activations remain floating-point in the case of weights compression which leads to a better accuracy. Weight compression for LLMs provides a solid inference performance improvement which is on par with the performance of the full model quantization. In addition, weight compression is data-free and does not require a calibration dataset, making it easy to use.

`nncf.compress_weights` function can be used for performing weights compression. The function accepts an OpenVINO model and other compression parameters. Compared to INT8 compression, INT4 compression improves performance even more, but introduces a minor drop in prediction quality.

More details about weights compression, can be found in [OpenVINO documentation](https://docs.openvino.ai/2024/openvino-workflow/model-optimization-guide/weight-compression.html).
</details>

In [4]:
import ipywidgets as widgets

model_format = widgets.Dropdown(
    options=["FP16", "INT8", "INT4"],
    value="INT4",
    description="Model format:",
)

model_id = "OpenGVLab/InternVideo2-Stage2_6B"
model_format

Dropdown(description='Model format:', options=('FP16', 'INT8', 'INT4'), value='FP16')

In [5]:
import nncf

base_model_dir = Path(model_id.split("/")[-1] + "_ov")
model_dir = base_model_dir / model_format.value

if model_format.value == "INT4":
    weights_compression_config = {"mode": nncf.CompressWeightsMode.INT4_ASYM, "group_size": 128, "ratio": 1.0}
elif model_format.value == "INT8":
    weights_compression_config = {"mode": nncf.CompressWeightsMode.INT8_ASYM}
else:
    weights_compression_config = None


convert_internvideo(model_id, model_dir, weights_compression_config)

/home/ea/work/my_optimum_intel/optimum_env_new/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/ea/.cache/huggingface/modules/transformers_modules/InternVideo2-Stage2_6B/modeling_internvideo2.py:508: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/home/ea/work/my_optimum_intel/optimum_env_new/lib/python3.11/site-packages/transformers/configuration_utils.py:311: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.w

FusedMLP of flash_attn is not installed!!!
DropoutAddRMSNorm of flash_attn is not installed!!!


BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of the model checkpoint at bert-large-uncased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.bias', 'cls.seq_relationship.weig

Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

Some weights of InternVideo2_Stage2 were not initialized from the model checkpoint at InternVideo2-Stage2_6B and are newly initialized: ['text_encoder.cls.predictions.decoder.bias', 'text_encoder.cls.predictions.decoder.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model successfully converted


## Prepare model inference pipeline
[back to top ⬆️](#Table-of-contents:)


`OVInternVideoStage2` defined in `ov_internvideo_helper.py` provides unified interface for running model inference. It accepts model directory and target device map for inference.

In [6]:
from ov_internvideo_helper import OVInternVideoStage2

# Uncomment the line to see model inference code
# ??OVInternVidoeStage2

### Select inference device
[back to top ⬆️](#Table-of-contents:)

In [7]:
from notebook_utils import device_widget


device_vision = device_widget(exclude=["NPU"], description="Vision Device")
device_text = device_widget(exclude=["NPU"], description="Text Device")

devices = widgets.VBox([device_vision, device_text])

devices

In [8]:
ov_model = OVInternVideoStage2(model_dir, device_map={"text_encoder": device_text.value, "vision_encoder": device_vision.value})

FusedMLP of flash_attn is not installed!!!
DropoutAddRMSNorm of flash_attn is not installed!!!


/home/ea/.cache/huggingface/modules/transformers_modules/FP16/modeling_internvideo2.py:508: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


## Run model inference
[back to top ⬆️](#Table-of-contents:)


Now, let's see model in action. We will use `retrieve_text` helper function for finding the most relevant caption for video in the same way like demonstrated in original model example of usage.

In [9]:
from notebook_utils import download_file

if not Path("coco.mp4").exists():
    download_file(
        "https://storage.openvinotoolkit.org/repositories/openvino_notebooks/data/data/video/Coco%20Walking%20in%20Berkeley.mp4",
        filename="coco.mp4",
    )

widgets.Video.from_file("coco.mp4", loop=False, width=800, height=800)

Video(value=b'\x00\x00\x00\x18ftypmp42\x00\x00\x00\x00isommp42\x00\x00\x18\tmoov\x00\x00\x00lmvhd...', height=…

In [10]:
import cv2

from ov_internvideo_helper import retrieve_text, _frame_from_video

text_candidates = ["A dog", "A bird", "A cat"]

video = cv2.VideoCapture("coco.mp4")
frames = [x for x in _frame_from_video(video)]

texts, probs = retrieve_text(frames, text_candidates, vlm=ov_model, topk=3)
for t, p in zip(texts, probs):
    print(f"text: {t} ~ prob: {p:.4f}")

text: A dog ~ prob: 0.7721
text: A bird ~ prob: 0.1732
text: A cat ~ prob: 0.0547


## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo


def run(video_file, labels):
    video = cv2.VideoCapture(video_file)
    frames = [x for x in _frame_from_video(video)]
    text_candidates = labels.split(",")
    top_k = min(len(text_candidates), 5)
    texts, probs = retrieve_text(frames, text_candidates, vlm=ov_model, topk=top_k)
    return {label: float(prob) for label, prob in zip(texts, probs)}


demo = make_demo(run)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/